# Limpieza de los Datos de la Tabla bronce.sucursales para Cargalos en la Capa Plata

Proposito del script:  
- Verificar columna por columna los tipos de datos para encontrar inconsistencias en los datos.  
- Limpiar y estandarizar, columna por columna los datos.  
- Exportar la nueva tabla como un archivo con nombre "semi_limpio_sucursales.parquet".

# Estableciendo Conexion

In [1]:
# Importando librerias y estableciendo conexion 
import pandas as pd 
from datetime import date
from funciones import limpiar_texto, formato_estado_sucursal, formato_zona
from conexiones_y_rutas import obtener_engine, obtener_ruta_archivo
engine = obtener_engine()

df_sucursales = pd.read_sql(
    "SELECT * FROM bronce.sucursales",
    con=engine
)

df_sucursales_tra = df_sucursales.copy()

# Archivos de Ayuda

**Nota**: Este archivo se va utilizar para verificar las siguientes columnas:  
- [Ciudad](#ciudad)
- [Departamento](#departamento)
- [Region](#region)

In [2]:
# Importa ciudades, departamentos y regiones del peru 
df_ciuda = pd.read_csv(obtener_ruta_archivo("archivos_de_ayuda","peru_ciudades.csv"))
df_ciudades = df_ciuda.copy()

# Resumen de las Columnas

- **sucursal_id**: Identificador unico de cada sucursal.  
- **codigo_sucursal**: Codigo unico de cada sucursal SUC00(sucursal_id) => 7 digitos maximo.  
- **nombre_sucursal**: Nombre de la sucursal.  
- **tipo_sucursal**: Tipo de sucursal (ejem: Oficina Principal y Agencia).  
- **ciudad**: Ciudad donde se ubica dicha sucursal (ejem: Miraflores y Santiago de Surco).  
- **departamento**: Departamento donde se ubica dicha sucursal (ejem: Lima e Ica).  
- **region**: Region donde se ubica dicha sucursal (ejem: Lima y Callao).  
- **zona**: Zona donde se ubica dicha sucursal (ejem: Urbano y Rural).  
- **fecha_apertura**: Fecha de apertura de la sucursal.
- **estado_sucursal**: Estado actual de la sucursal (ejem: Activo e Inactivo).

# Verificacion de la Calidad y Limpieza de los Datos

In [3]:
df_sucursales_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sucursal_id      26 non-null     int64 
 1   codigo_sucursal  23 non-null     object
 2   nombre_sucursal  26 non-null     object
 3   tipo_sucursal    26 non-null     object
 4   ciudad           26 non-null     object
 5   departamento     26 non-null     object
 6   region           26 non-null     object
 7   zona             26 non-null     object
 8   fecha_apertura   26 non-null     object
 9   estado_sucursal  26 non-null     object
dtypes: int64(1), object(9)
memory usage: 2.2+ KB


In [4]:
df_sucursales_tra.head()

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
0,1,SUC0001,Oficina Principal Lima,OFICINA PRINCIPAL,San Juan de Lurigancho,Lima,Lima y Callao,Urbano,2005-07-24,Activa
1,2,SUC0002,Agencia Santiago de Surco 1,Agencia,Santiago de Surco,Lima,Lima y Callao,Urbano,2010-01-03,ACTIVA
2,3,SUC0003,Agencia Miraflores 2,Punto de Atención,MiRaFlOrEs,Lima,Lima y Callao,Urbano,2007-04-20,Activa
3,4,SUC0004,Agencia Los Olivos 3,Agencia,Los Olivos,Lima,Lima y Callao,urbano,2014/06/19,Activa
4,5,SUC0005,Agencia Lima 4,Agencia,Lima,Lima,Lima y Callao,Urbano,2007/02/07,Activa


In [5]:
# Verifica si existen registros duplicados 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.duplicated(keep=False)]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
20,21,None,Oficina Principal Ancash,Oficina Principal,Chimbote,AnCaSh,Centro,Urbano,2006-11-01,Activa
24,21,None,Oficina Principal Ancash,Oficina Principal,Chimbote,AnCaSh,Centro,Urbano,2006-11-01,Activa


In [6]:
# Elimina duplicados
df_sucursales_tra.drop_duplicates(inplace=True)
df_sucursales_tra.reset_index(drop=True,inplace=True)

## sucursal_id

In [7]:
# Verifica si existen ids negativos o 0 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.sucursal_id <= 0]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal


**Nota**: Primero se va a limpiar las columnas y las fechas para verificar que fecha es la mas reciente.  
La limpieza se va a realizar en:  
[sucursal_id V2](#sucursal_id-v2)

In [8]:
# Muestra identificadores duplicados 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.sucursal_id.duplicated(keep=False)]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
19,20,SUC0020,Oficina Principal Ica,OFICINA PRINCIPAL,Pisco,Ica,SuR,Urbano,2009/04/25,Activa
24,20,SUC0020,Oficina Principal Ica,OFICINA PRINCIPAL,Pisco,Ica,SuR,Urbano,2009-04-25,Activa


## codigo_sucursal

In [9]:
# Verifica si los formatos son correctos 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra.codigo_sucursal[
    df_sucursales_tra.codigo_sucursal 
        != df_sucursales_tra.codigo_sucursal.str.strip().str.upper()
]

13    None
20    None
Name: codigo_sucursal, dtype: object

In [10]:
# Estandariza los formatos
df_sucursales_tra["codigo_sucursal"] = df_sucursales_tra.codigo_sucursal.apply(
    lambda cod_sucursal: cod_sucursal.strip().upper()
    if pd.notna(cod_sucursal)
    else 'n/a'
)

In [11]:
# Verifica si existe duplicados 
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.codigo_sucursal.duplicated(keep=False)]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
13,14,n/a,Oficina Principal Piura,Oficina Principal,Piura,Piura,Norte,Urbano,19-01-2013,Activa
19,20,SUC0020,Oficina Principal Ica,OFICINA PRINCIPAL,Pisco,Ica,SuR,Urbano,2009/04/25,Activa
20,21,n/a,Oficina Principal Ancash,Oficina Principal,Chimbote,AnCaSh,Centro,Urbano,2006-11-01,Activa
24,20,SUC0020,Oficina Principal Ica,OFICINA PRINCIPAL,Pisco,Ica,SuR,Urbano,2009-04-25,Activa


In [12]:
# Recrea los codigos de sucursal, utilizando la logica observada
cod_sucur = (
    "SUC"
    + df_sucursales_tra["sucursal_id"]
        .astype(str)
        .str.zfill(4)
)
# Verifica si los codigos generados son iguales a los codigos existentes
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.codigo_sucursal != cod_sucur]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
13,14,n/a,Oficina Principal Piura,Oficina Principal,Piura,Piura,Norte,Urbano,19-01-2013,Activa
20,21,n/a,Oficina Principal Ancash,Oficina Principal,Chimbote,AnCaSh,Centro,Urbano,2006-11-01,Activa


In [13]:
# Reemplaza los codigos 
df_sucursales_tra["codigo_sucursal"] = cod_sucur
# Verifica si los codigos generados son iguales a los codigos existentes
# Resultados Esperados: Tabla Vacia
df_sucursales_tra[df_sucursales_tra.codigo_sucursal != cod_sucur]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal


## nombre_sucursal

**Nota**: Tengo dudas sobre el nombre, porque, parecer ser una union de varias columnas:  
- Si es oficina principal es tipo_sucursal + departamento.  
- Si es agencia es tipo_sucursal + ciudad.  
Pero tambien existen "oficinas especiales" y "puntos de atencion" y estos valores no siguen esta regla, asi que no estoy muy seguro, como no puedo resolver esta duda voy a dejar los nombres como estan.

In [14]:
# Verifica el formato de los nombres 
# Resultados Esperados: Tabla Vacia 
df_sucursales_tra.nombre_sucursal[df_sucursales_tra.nombre_sucursal 
                                != df_sucursales_tra.nombre_sucursal.str.strip().str.title()]

0             Oficina Principal Lima 
1         Agencia Santiago de Surco 1
5          OfIcInA PrInCiPaL ArEqUiPa
8       oficina principal la libertad
10     Agencia Víctor Larco Herrera 2
Name: nombre_sucursal, dtype: object

In [15]:
# Estandariza el formato del texto 
df_sucursales_tra["nombre_sucursal"] = df_sucursales_tra.nombre_sucursal.apply(limpiar_texto)
# Verifica el formato de los nombres 
# Resultados Esperados: Agencia Santiago de Surco 1, Agencia los Olivos 3,  Oficina Principal la Libertad, Agencia el Tambo 1
df_sucursales_tra.nombre_sucursal[df_sucursales_tra.nombre_sucursal 
                                != df_sucursales_tra.nombre_sucursal.str.strip().str.title()]

1       Agencia Santiago de Surco 1
3              Agencia los Olivos 3
8     Oficina Principal la Libertad
18               Agencia el Tambo 1
Name: nombre_sucursal, dtype: object

## tipo_sucursal

In [16]:
# Resultados Esperados: 'Oficina Principal', 'Agencia', 'Punto de Atención', 'Oficina Especial'
df_sucursales_tra.tipo_sucursal.unique()

array(['OFICINA PRINCIPAL', 'Agencia', 'Punto de Atención',
       'Oficina Principal', 'Oficina Especial', 'Agencia ', 'AGENCIA',
       'oficina principal'], dtype=object)

In [17]:
# Resultados Esperados: Tabla Vacia 
df_sucursales_tra.tipo_sucursal[df_sucursales_tra.tipo_sucursal 
                                != df_sucursales_tra.tipo_sucursal.str.strip().str.title()]

0     OFICINA PRINCIPAL
2     Punto de Atención
6     Punto de Atención
12             Agencia 
14    Punto de Atención
16              AGENCIA
18    Punto de Atención
19    OFICINA PRINCIPAL
23    oficina principal
24    OFICINA PRINCIPAL
Name: tipo_sucursal, dtype: object

In [18]:
# Estandariza el formato del texto 
# Resultados Esperados: Punto de Atención
df_sucursales_tra["tipo_sucursal"] = df_sucursales_tra.tipo_sucursal.apply(limpiar_texto)
df_sucursales_tra.tipo_sucursal[df_sucursales_tra.tipo_sucursal 
                                != df_sucursales_tra.tipo_sucursal.str.strip().str.title()]

2     Punto de Atención
6     Punto de Atención
14    Punto de Atención
18    Punto de Atención
Name: tipo_sucursal, dtype: object

In [19]:
df_sucursales_tra.tipo_sucursal.unique()

array(['Oficina Principal', 'Agencia', 'Punto de Atención',
       'Oficina Especial'], dtype=object)

## ciudad

In [20]:
# Verifica si las ciudades siguen el formato adecuado 
# Resultados Esperados: Tabla Vacia 
df_sucursales_tra.ciudad[df_sucursales_tra.ciudad 
                        != df_sucursales_tra.ciudad.str.strip().str.title()]

0       San Juan de Lurigancho
1            Santiago de Surco
2                   MiRaFlOrEs
10        VícTor laRco heRreRa
12                    Chiclayo
16                     WaNcHaQ
21                     IQUITOS
23                   CAJAMARCA
Name: ciudad, dtype: object

In [21]:
# Estandariza el formato del texto 
# Resultados Esperados: San Juan de Lurigancho, Santiago de Surco
df_sucursales_tra["ciudad"] = df_sucursales_tra.ciudad.apply(limpiar_texto)
df_sucursales_tra.ciudad[df_sucursales_tra.ciudad 
                                != df_sucursales_tra.ciudad.str.strip().str.title()]

0    San Juan de Lurigancho
1         Santiago de Surco
Name: ciudad, dtype: object

In [22]:
# Valida si todas las ciudades de sucursales, se encuentra en el registro de la empresa 
# Resultados esperados: both: 25, left_only: 0, right_only: 0
verficar_ciudad = df_sucursales_tra.merge(
    right=df_ciudades,
    on="ciudad",
    indicator=True,
    suffixes=["_sucur_tra","_ciudades"]
)
verficar_ciudad._merge.value_counts()

_merge
both          25
left_only      0
right_only     0
Name: count, dtype: int64

## departamento

In [23]:
# Verifica si los departamentos siguen el formato adecuado 
# Resultados Esperados: Tabla Vacia 
df_sucursales_tra.departamento[df_sucursales_tra.departamento 
                            != df_sucursales_tra.departamento.str.strip().str.title()]

3            Lima
10    la libertad
18          Junín
19            Ica
20         AnCaSh
24            Ica
Name: departamento, dtype: object

In [24]:
# Estandariza el formato de los departamentos 
df_sucursales_tra["departamento"] = df_sucursales_tra.departamento.apply(limpiar_texto)
# Verifica si los departamentos siguen el formato adecuado 
# Resultados Esperados: Tabla Vacia 
df_sucursales_tra.departamento[df_sucursales_tra.departamento 
                                != df_sucursales_tra.departamento.str.strip().str.title()]

Series([], Name: departamento, dtype: object)

In [25]:
# Valida si para la ciudad el departamento es el correcto 
# Resultados esperados: Tabla Vacía
verificar_departamento = df_sucursales_tra.merge(right=df_ciudades,
                                                on="ciudad",
                                                suffixes=["_sucur_tra","_ciudades"]
                                            )
verificar_departamento[verificar_departamento.departamento_sucur_tra != verificar_departamento.departamento_ciudades]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento_sucur_tra,region_sucur_tra,zona,fecha_apertura,estado_sucursal,departamento_ciudades,region_ciudades
20,21,SUC0021,Oficina Principal Ancash,Oficina Principal,Chimbote,Ancash,Centro,Urbano,2006-11-01,Activa,Áncash,Centro


In [26]:
# Reemplaza por el nombre correcto del departamento
df_sucursales_tra.loc[df_sucursales_tra.departamento == "Ancash","departamento"] = "Áncash"

## region

In [27]:
# Valida si la region tiene el formato correcto
# Resultados esperados: Tabla Vacía 
df_sucursales_tra.region[df_sucursales_tra.region 
                        != df_sucursales_tra.region.str.strip().str.title()]

0     Lima y Callao
1     Lima y Callao
2     Lima y Callao
3     Lima y Callao
4     Lima y Callao
5              Sur 
19              SuR
24              SuR
Name: region, dtype: object

In [28]:
# Estandariza el formato de la region 
# Resultados Esperados: Lima y Callao
df_sucursales_tra["region"] = df_sucursales_tra.region.apply(limpiar_texto)
df_sucursales_tra.region[df_sucursales_tra.region 
                                != df_sucursales_tra.region.str.strip().str.title()]

0    Lima y Callao
1    Lima y Callao
2    Lima y Callao
3    Lima y Callao
4    Lima y Callao
Name: region, dtype: object

In [29]:
# Valida si para la ciudad la region es la correcta 
# Resultados esperados: Tabla Vacia
verificar_region = df_sucursales_tra.merge(right=df_ciudades,
                                        on="ciudad",
                                        suffixes=["_sucur_tra","_ciudades"]
                                    )
verificar_region[verificar_region.region_sucur_tra != verificar_region.region_ciudades]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento_sucur_tra,region_sucur_tra,zona,fecha_apertura,estado_sucursal,departamento_ciudades,region_ciudades


## zona

In [30]:
# Resultados Esperados: 'Urbano', 'Selva','Rural', 'n/a'
df_sucursales_tra.zona.unique()

array(['Urbano', 'urbano', 'URBANO', 'Selva'], dtype=object)

In [31]:
# Estandariza el formato de la zona 
# Resultados Esperados: 'Urbano', 'Selva','Rural', 'n/a'
df_sucursales_tra["zona"] = df_sucursales_tra.zona.apply(formato_zona)
df_sucursales_tra.zona.unique()

array(['Urbano', 'Selva'], dtype=object)

## fecha_apertura

In [32]:
# Verifica si existen fechas que generen errores 
# Resultado Esperado: Tabla Vacia
fechas_apertura_error = pd.to_datetime(
    df_sucursales_tra.fecha_apertura,
    errors='coerce'
)
df_sucursales_tra.fecha_apertura[fechas_apertura_error.isna()]

3     2014/06/19
4     2007/02/07
6     2009/06/17
7     28-01-2015
8     29/03/2011
9     25/06/2014
10    2008/06/27
11    20-07-2012
12    2013/07/10
13    19-01-2013
14    2010/12/07
15    22/04/2015
16    28/06/2013
17    30/07/2011
19    2009/04/25
21    29-06-2011
22    2010/03/23
Name: fecha_apertura, dtype: object

In [33]:
# Cambiando a tipo fecha usando un formato mixed
df_sucursales_tra["fecha_apertura"] = pd.to_datetime(
    df_sucursales_tra.fecha_apertura,
    errors='coerce',
    format='mixed',
    dayfirst= True
)
df_sucursales_tra.fecha_apertura[fechas_apertura_error.isna()]

3    2014-06-19
4    2007-02-07
6    2009-06-17
7    2015-01-28
8    2011-03-29
9    2014-06-25
10   2008-06-27
11   2012-07-20
12   2013-07-10
13   2013-01-19
14   2010-12-07
15   2015-04-22
16   2013-06-28
17   2011-07-30
19   2009-04-25
21   2011-06-29
22   2010-03-23
Name: fecha_apertura, dtype: datetime64[ns]

In [34]:
# Verifica el estado de las fechas que tenian error al incio 
df_sucursales_tra.fecha_apertura[fechas_apertura_error.isna()]

3    2014-06-19
4    2007-02-07
6    2009-06-17
7    2015-01-28
8    2011-03-29
9    2014-06-25
10   2008-06-27
11   2012-07-20
12   2013-07-10
13   2013-01-19
14   2010-12-07
15   2015-04-22
16   2013-06-28
17   2011-07-30
19   2009-04-25
21   2011-06-29
22   2010-03-23
Name: fecha_apertura, dtype: datetime64[ns]

In [35]:
# Verifica que las fechas de apertura no sean futuras 
# Resultados Esperados: Tabla Vacias
df_sucursales_tra[df_sucursales_tra.fecha_apertura.dt.date >= date.today()]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal


## estado_sucursal

In [36]:
# Resultados Esperados: 'Activa', 'Inactiva', 'n/a'
df_sucursales_tra.estado_sucursal.unique()

array(['Activa', 'ACTIVA', 'activa', 'Activa '], dtype=object)

In [37]:
# Resultados Esperados: 'Activa', 'Inactiva', 'n/a'
df_sucursales_tra["estado_sucursal"] = df_sucursales_tra.estado_sucursal.apply(formato_estado_sucursal)
df_sucursales_tra.estado_sucursal.unique()

array(['Activa'], dtype=object)

# Limpiando Duplicados Luego de Limpieza

In [38]:
# Verifica si existen registros duplicados luego de la limpieza
# Resultados Esperados: Tabla Vacias
df_sucursales_tra[df_sucursales_tra.duplicated(keep=False)]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
19,20,SUC0020,Oficina Principal Ica,Oficina Principal,Pisco,Ica,Sur,Urbano,2009-04-25,Activa
24,20,SUC0020,Oficina Principal Ica,Oficina Principal,Pisco,Ica,Sur,Urbano,2009-04-25,Activa


In [39]:
df_sucursales_tra.drop_duplicates(inplace=True)
df_sucursales_tra.reset_index(drop=True,inplace=True)

## sucursal_id V2

In [40]:
# Verifica si existen ids duplicados
# Resultados Esperados: Tabla Vacias
df_sucursales_tra[df_sucursales_tra.sucursal_id.duplicated(keep=False)]

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal


# Exportando la Tabla Limpia

In [41]:
df_sucursales_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   sucursal_id      24 non-null     int64         
 1   codigo_sucursal  24 non-null     object        
 2   nombre_sucursal  24 non-null     object        
 3   tipo_sucursal    24 non-null     object        
 4   ciudad           24 non-null     object        
 5   departamento     24 non-null     object        
 6   region           24 non-null     object        
 7   zona             24 non-null     object        
 8   fecha_apertura   24 non-null     datetime64[ns]
 9   estado_sucursal  24 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(8)
memory usage: 2.0+ KB


In [42]:
df_sucursales_tra.head()

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
0,1,SUC0001,Oficina Principal Lima,Oficina Principal,San Juan de Lurigancho,Lima,Lima y Callao,Urbano,2005-07-24,Activa
1,2,SUC0002,Agencia Santiago de Surco 1,Agencia,Santiago de Surco,Lima,Lima y Callao,Urbano,2010-01-03,Activa
2,3,SUC0003,Agencia Miraflores 2,Punto de Atención,Miraflores,Lima,Lima y Callao,Urbano,2007-04-20,Activa
3,4,SUC0004,Agencia los Olivos 3,Agencia,Los Olivos,Lima,Lima y Callao,Urbano,2014-06-19,Activa
4,5,SUC0005,Agencia Lima 4,Agencia,Lima,Lima,Lima y Callao,Urbano,2007-02-07,Activa


In [43]:
df_sucursales_tra.to_parquet(
    obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_sucursales.parquet"),
    index=False
)